# 🔥 Ternary LLM — 2-bit Inference Engine

This notebook demonstrates:
1. Building the C++ / CUDA inference engine
2. Ternarizing a HuggingFace LLM to 2-bit weights
3. Running inference with custom CUDA kernels

**Requirements:** GPU runtime (Runtime → Change runtime type → T4 GPU)

## Step 1: Clone & Build

In [ ]:
# === CHANGE THIS to your GitHub repo URL ===
REPO_URL = "https://github.com/YOUR_USERNAME/ternary-llm.git"

!git clone {REPO_URL} /content/ternary-llm 2>/dev/null || echo 'Already cloned'
%cd /content/ternary-llm

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!nvcc --version | grep release

In [ ]:
# Build the C++ engine
!mkdir -p build && cd build && \
    cmake .. -DCMAKE_BUILD_TYPE=Release \
             -DCMAKE_CUDA_ARCHITECTURES="75" \
    && make -j$(nproc)

# Note: use 75 for T4, 80 for A100, 89 for L4
# Check your GPU with: nvidia-smi --query-gpu=compute_cap --format=csv

In [ ]:
# Verify build
!ls -lh build/ternary-llm build/test_gemv build/benchmark

## Step 2: Run Kernel Tests

In [ ]:
# Test GEMV kernel correctness
!./build/test_gemv

In [ ]:
# Benchmark: ternary GEMV vs FP16 GEMV
!./build/benchmark

## Step 3: Ternarize a Model

We'll use a small model for demonstration. You can change to any LLaMA/Mistral model.

In [ ]:
# Install Python dependencies
!pip install -q torch transformers safetensors sentencepiece numpy tqdm

In [ ]:
# Ternarize the model (this downloads the model from HuggingFace)
# For Colab free tier, use a small model (1B-3B).
# For Colab Pro (A100), you can try 7B-8B.

MODEL_NAME = "meta-llama/Llama-3.2-1B"  # <-- change as needed
OUTPUT_FILE = "/content/ternary-llm/model.tllm"

!cd /content/ternary-llm/ternarize && python ternarize.py \
    --model {MODEL_NAME} \
    --output {OUTPUT_FILE} \
    --group-size 128 \
    --threshold-factor 0.7

In [ ]:
# Check the output size
import os
size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"Ternary model size: {size_mb:.1f} MB")

## Step 4: Run Inference

In [ ]:
# Generate text with benchmark stats
!./build/ternary-llm \
    --model model.tllm \
    --prompt "The meaning of life is" \
    --max-tokens 100 \
    --temperature 0.7 \
    --top-p 0.9 \
    --benchmark

In [ ]:
# Try different prompts
prompts = [
    "Once upon a time in a land far away,",
    "The theory of general relativity states that",
    "def fibonacci(n):",
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt}")
    print('='*60)
    !./build/ternary-llm --model model.tllm --prompt "{prompt}" --max-tokens 80 --benchmark

## Step 5: Memory Analysis

In [ ]:
# Show GPU memory usage
!nvidia-smi

## Packing Tests (Python)

In [ ]:
# Run the Python packing correctness tests
!cd /content/ternary-llm && python tests/test_packing.py

---

## Notes

- **T4 GPU (Colab Free):** `CMAKE_CUDA_ARCHITECTURES=75`, works well with 1B-3B models
- **A100 GPU (Colab Pro):** `CMAKE_CUDA_ARCHITECTURES=80`, can handle 7B-8B models
- **L4 GPU (Colab Pro):** `CMAKE_CUDA_ARCHITECTURES=89`
- For gated models (LLaMA), you need a HuggingFace token:
  ```python
  from huggingface_hub import login
  login(token='hf_YOUR_TOKEN')
  ```